# 07 — Intervention simulation (pipeline)

Parametric counterfactual: shift accessibility posteriors for the top-*N* tracts identified
under two targeting rules (Bayesian exceedance vs deterministic jobs count) and compare the
resulting equity impact. Ground the shift in GTFS headway data; propagate through all
posterior draws; export Fig 1 (study area), Fig 9 (intervention effect) and the dashboard
GeoJSONs defined in `context/INTERFACES.md`.

**Previous:** [`06_equity_decomposition.ipynb`](06_equity_decomposition.ipynb)
**Next:** *(end of pipeline — paper writing + `app/` dashboard)*

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import yaml


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "configs" / "san_diego.yaml").exists():
            return d
    raise FileNotFoundError(
        "Could not find configs/san_diego.yaml. cd to the repo root or start Jupyter there."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / "configs" / "san_diego.yaml", encoding="utf-8") as f:
    CITY = yaml.safe_load(f)
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"city = {CITY.get('city_label') or CITY.get('city')!r}, bbox = {CITY.get('bbox')}")

REPO_ROOT = C:\Users\sardo\OneDrive\Desktop\Classes\Projects\BayesTransitEquity
city = 'San Diego, CA', bbox = [-117.4, 32.53, -116.8, 33.35]


## 1 — Environment setup and configuration

In [2]:
RID = "fit_raw_zscore_x"
N_INTERVENTION_TRACTS = 20
INTERVENTION_STRENGTH = 0.5
INTERVENTION_STRENGTH_GRID = (0.25, 0.5, 0.75)
SCENARIO_A_ID = "freq_double_bayesian_top20"
SCENARIO_B_ID = "freq_double_det_top20"
RNG_SEED = 20260417
HOOK_GEOID = "06073013317"
TRAVEL_TIME_MIN = int(CITY.get("accessibility", {}).get("travel_time_threshold_min", 45))
N_BOOT = 400
SPILLOVER_WEIGHT = 0.1
WELL_SERVED_EXCEEDANCE_THR = 0.05
DESERT_EXCEEDANCE_THR = 0.5

In [3]:
import json
from datetime import date

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import spearmanr

PIPE_TABLES = REPO_ROOT / "artifacts" / "tables" / "pipeline"
FIGDIR = REPO_ROOT / "artifacts" / "figures"
POSTDIR = REPO_ROOT / "data" / "processed" / "posteriors"
GEOJSON_DIR = REPO_ROOT / "data" / "processed" / "geojson"
SCENARIO_DIR = GEOJSON_DIR / "scenarios"
for d in (PIPE_TABLES, FIGDIR, GEOJSON_DIR, SCENARIO_DIR):
    d.mkdir(parents=True, exist_ok=True)

TODAY = date.today().isoformat()
rng = np.random.default_rng(RNG_SEED)


def pipe_tbl(stem: str) -> Path:
    return PIPE_TABLES / f"pipeline__07_{stem}__{RID}.csv"


def pipe_fig(stem: str, *, with_rid: bool = True) -> Path:
    suffix = f"__{RID}" if with_rid else ""
    return FIGDIR / f"pipeline__07_{stem}{suffix}.png"

In [4]:
summary_path = POSTDIR / f"{RID}_posterior_summary.parquet"
samples_path = POSTDIR / f"{RID}_posterior_samples.parquet"

post_summary = pd.read_parquet(summary_path)
post_summary["GEOID"] = post_summary["GEOID"].astype(str).str.zfill(11)

samples_long = pd.read_parquet(samples_path)
samples_long["geoid"] = samples_long["geoid"].astype(str).str.zfill(11)
draws_per_chain = int(samples_long["sample_idx"].max()) + 1
samples_long["draw_id"] = samples_long["chain"] * draws_per_chain + samples_long["sample_idx"]

wide = samples_long.pivot(index="geoid", columns="draw_id", values="accessibility").sort_index()
geoid_order = wide.index.to_numpy()
geoid_to_idx = {g: i for i, g in enumerate(geoid_order)}
samples_wide = wide.to_numpy(dtype=np.float64, copy=True)
n_tracts, n_draws = samples_wide.shape
print(f"samples_wide shape: {samples_wide.shape}  log1p range=({samples_wide.min():.2f}, {samples_wide.max():.2f})")

samples_wide shape: (720, 32000)  log1p range=(3.81, 12.95)


In [5]:
tract_zip = REPO_ROOT / "data" / "raw" / "census" / "tl_2023_06_tract.zip"
tracts_gdf = gpd.read_file(f"zip://{tract_zip}")
tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str).str.zfill(11)
tracts_gdf = tracts_gdf[tracts_gdf["COUNTYFP"] == "073"].copy()

acs_candidates = sorted((REPO_ROOT / "artifacts" / "tables" / "eda").glob("eda__acs_sd_tract_attributes__*.csv"))
acs = pd.read_csv(acs_candidates[-1], dtype={"GEOID": str})
acs["GEOID"] = acs["GEOID"].str.zfill(11)

acc_bundle = pd.read_parquet(
    REPO_ROOT / "data" / "processed" / "accessibility" / "tract_accessibility_bundle__2026-04-03.parquet"
)
acc_bundle["GEOID"] = acc_bundle["GEOID"].astype(str).str.zfill(11)

print(f"tracts: {len(tracts_gdf)} (county 073)   acs: {len(acs)}   accessibility rows: {len(acc_bundle)}")

tracts: 737 (county 073)   acs: 737   accessibility rows: 726


In [6]:
jobs_col = f"jobs_C000_{TRAVEL_TIME_MIN}min"
exc_col = f"exceedance_prob_{TRAVEL_TIME_MIN}min"

master = (
    post_summary[["GEOID", "posterior_mean_log1p", "posterior_sd_log1p",
                   "posterior_mean_jobs", exc_col, "B01003_001E"]]
    .merge(acc_bundle[["GEOID", jobs_col]], on="GEOID", how="left")
    .merge(acs[["GEOID", "disadvantage_z", "no_vehicle_hh_rate", "poverty_rate"]], on="GEOID", how="left")
    .rename(columns={jobs_col: "det_jobs", exc_col: "exceedance_prob", "B01003_001E": "pop_total"})
)
master = master[master["GEOID"].isin(geoid_order)].reset_index(drop=True)

Q25_JOBS = float(acc_bundle[jobs_col].quantile(0.25))
Q25_LOG1P = float(np.log1p(Q25_JOBS))
print(f"Q25 jobs ({TRAVEL_TIME_MIN} min) = {Q25_JOBS:.0f}   log1p(Q25) = {Q25_LOG1P:.3f}   master rows: {len(master)}")

Q25 jobs (45 min) = 4470   log1p(Q25) = 8.405   master rows: 720


## 2 — Target selection: Bayesian vs deterministic

In [7]:
bayes_targets = master.nlargest(N_INTERVENTION_TRACTS, "exceedance_prob").copy()
bayes_targets["method"] = "bayesian"
det_targets = master.nsmallest(N_INTERVENTION_TRACTS, "det_jobs").copy()
det_targets["method"] = "deterministic"

overlap = sorted(set(bayes_targets.GEOID) & set(det_targets.GEOID))
only_bayes = sorted(set(bayes_targets.GEOID) - set(det_targets.GEOID))
only_det = sorted(set(det_targets.GEOID) - set(bayes_targets.GEOID))

targets = pd.concat([bayes_targets, det_targets], ignore_index=True)
targets["rank_in_method"] = targets.groupby("method").cumcount() + 1
targets["selected_by_bayes"] = targets["GEOID"].isin(bayes_targets.GEOID)
targets["selected_by_det"] = targets["GEOID"].isin(det_targets.GEOID)
targets["in_overlap"] = targets["GEOID"].isin(overlap)
targets = targets[["method", "rank_in_method", "GEOID", "exceedance_prob", "det_jobs",
                    "posterior_mean_jobs", "disadvantage_z", "pop_total",
                    "selected_by_bayes", "selected_by_det", "in_overlap"]]
targets.to_csv(pipe_tbl("intervention_targets"), index=False)
print(f"overlap={len(overlap)}  bayes_only={len(only_bayes)}  det_only={len(only_det)}"
      f"  hook_in_bayes={HOOK_GEOID in bayes_targets.GEOID.tolist()}")

overlap=6  bayes_only=14  det_only=14  hook_in_bayes=False


## 3 — GTFS coverage of target tracts + empirical calibration

Compute AM-peak (07:00–09:00) headways for MTS routes serving each target tract
(400 m buffer around tract polygon). Use the empirical Spearman rho(headway, det_jobs)
across all tracts to sanity-check that headway shortening corresponds to accessibility
gains in observed data — this grounds the parametric Δ defined in §4.

In [8]:
gtfs_mts = REPO_ROOT / "data" / "raw" / "gtfs" / "mts" / "extracted"
stops = pd.read_csv(gtfs_mts / "stops.txt", dtype=str, low_memory=False)
stop_times = pd.read_csv(gtfs_mts / "stop_times.txt", dtype=str, low_memory=False)
trips = pd.read_csv(gtfs_mts / "trips.txt", dtype=str, low_memory=False)

stops["stop_lon"] = pd.to_numeric(stops["stop_lon"])
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"])
stops_gdf = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs="EPSG:4326"
)


def hms_to_sec(x: str) -> int:
    h, m, s = x.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


AM_START, AM_END = 7 * 3600, 9 * 3600
st_dep = stop_times.dropna(subset=["departure_time"]).copy()
st_dep["dep_sec"] = st_dep["departure_time"].apply(hms_to_sec)
st_am = st_dep[(st_dep["dep_sec"] >= AM_START) & (st_dep["dep_sec"] <= AM_END)].copy()
st_am = st_am.merge(trips[["trip_id", "route_id"]], on="trip_id", how="left")
peak_min = (AM_END - AM_START) / 60.0
print(f"MTS stops: {len(stops_gdf)}   AM-peak stop_time rows: {len(st_am)}")

MTS stops: 4373   AM-peak stop_time rows: 80412


In [9]:
tracts_3857 = tracts_gdf[["GEOID", "geometry"]].to_crs(3857)
target_union = sorted(set(bayes_targets.GEOID) | set(det_targets.GEOID))

target_buffer = tracts_3857[tracts_3857.GEOID.isin(target_union)].copy()
target_buffer["geometry"] = target_buffer.geometry.buffer(400)
target_buffer_wgs = target_buffer.to_crs(4326)

stops_in_target = gpd.sjoin(
    stops_gdf, target_buffer_wgs[["GEOID", "geometry"]],
    predicate="intersects", how="inner"
).rename(columns={"GEOID": "target_GEOID"}).drop(columns="index_right")

st_am_near = st_am.merge(stops_in_target[["stop_id", "target_GEOID"]], on="stop_id", how="inner")
headways = (
    st_am_near.groupby(["target_GEOID", "route_id"])["trip_id"]
    .nunique().reset_index(name="n_trips_am_peak")
)
headways["headway_min"] = peak_min / headways["n_trips_am_peak"].clip(lower=1)


def freq_tier(h: float) -> str:
    if pd.isna(h):
        return "none"
    if h < 15:
        return "high"
    if h < 30:
        return "medium"
    return "low"


coverage = (
    headways.groupby("target_GEOID")
    .agg(n_routes=("route_id", "nunique"),
         min_headway_min=("headway_min", "min"),
         median_headway_min=("headway_min", "median"),
         total_trips_am=("n_trips_am_peak", "sum"))
    .reset_index()
    .rename(columns={"target_GEOID": "GEOID"})
)
coverage["current_frequency_tier"] = coverage["min_headway_min"].map(freq_tier)
coverage = coverage.merge(master[["GEOID", "exceedance_prob", "det_jobs", "disadvantage_z"]], on="GEOID", how="left")
coverage["selected_by_bayes"] = coverage["GEOID"].isin(bayes_targets.GEOID)
coverage["selected_by_det"] = coverage["GEOID"].isin(det_targets.GEOID)
coverage.to_csv(pipe_tbl("gtfs_coverage_targets"), index=False)
print(f"coverage rows: {len(coverage)}   with >=1 MTS route: {(coverage.n_routes > 0).sum()}")

coverage rows: 20   with >=1 MTS route: 20


In [10]:
all_buffer = tracts_3857.copy()
all_buffer["geometry"] = all_buffer.geometry.buffer(400)
stops_all = gpd.sjoin(
    stops_gdf, all_buffer.to_crs(4326)[["GEOID", "geometry"]],
    predicate="intersects", how="inner"
)
trips_per_tract = (
    st_am.merge(stops_all[["stop_id", "GEOID"]], on="stop_id", how="inner")
    .groupby("GEOID")["trip_id"].nunique()
    .reset_index(name="n_trips_am_peak")
)
trips_per_tract["min_headway_tract"] = peak_min / trips_per_tract["n_trips_am_peak"].clip(lower=1)

cal_df = master.merge(trips_per_tract, on="GEOID", how="left")
rho_head, p_head = spearmanr(cal_df["min_headway_tract"], cal_df["det_jobs"], nan_policy="omit")
rho_head = float(rho_head)
p_head = float(p_head)
print(f"Spearman(min_headway_tract, det_jobs) = {rho_head:.3f}  p={p_head:.2e}")

calibration = pd.DataFrame([
    {"metric": "spearman_headway_vs_det_jobs", "value": rho_head,
     "note": "AM-peak minimum headway within 400m buffer vs det jobs (all tracts). Negative → lower headway associated with higher access."},
    {"metric": "Q25_jobs", "value": float(Q25_JOBS), "note": f"25th percentile of jobs_C000_{TRAVEL_TIME_MIN}min"},
    {"metric": "Q25_log1p", "value": float(Q25_LOG1P), "note": "log1p(Q25_JOBS) — desert threshold on model scale"},
    {"metric": "n_targets_with_gtfs_coverage", "value": int((coverage.n_routes > 0).sum()),
     "note": "target tracts (union) with >=1 MTS route within 400m buffer"},
])
calibration.to_csv(pipe_tbl("calibration"), index=False)

Spearman(min_headway_tract, det_jobs) = -0.529  p=3.07e-37


## 4 — Parametric intervention model

Define a per-tract shift `Δ_log1p = strength × (well-served median log1p μ − target mean log1p μ)`.
`well_served` = median `posterior_mean_log1p` among tracts with baseline exceedance < 0.05.
Δ is clipped at 0 (never reduces access). Run a 3-point sensitivity grid on `strength`.

In [11]:
well_served_mask = master["exceedance_prob"] < WELL_SERVED_EXCEEDANCE_THR
well_served_log1p = float(master.loc[well_served_mask, "posterior_mean_log1p"].median())
print(f"well-served median log1p = {well_served_log1p:.3f}"
      f"  (n_well_served = {int(well_served_mask.sum())})")


def compute_delta(strength: float, target_ids: list[str]) -> pd.DataFrame:
    idx = master.set_index("GEOID").loc[target_ids]
    delta = np.maximum(strength * (well_served_log1p - idx["posterior_mean_log1p"].to_numpy()), 0.0)
    return pd.DataFrame({
        "GEOID": target_ids,
        "target_mean_log1p": idx["posterior_mean_log1p"].to_numpy(),
        "delta_log1p": delta,
        "delta_jobs": np.expm1(idx["posterior_mean_log1p"].to_numpy() + delta)
                      - np.expm1(idx["posterior_mean_log1p"].to_numpy()),
    })


delta_A = compute_delta(INTERVENTION_STRENGTH, bayes_targets.GEOID.tolist())
delta_B = compute_delta(INTERVENTION_STRENGTH, det_targets.GEOID.tolist())

sens_rows = []
for s in INTERVENTION_STRENGTH_GRID:
    for method, ids in [("bayesian", bayes_targets.GEOID.tolist()),
                         ("deterministic", det_targets.GEOID.tolist())]:
        d = compute_delta(s, ids)
        sens_rows.append({
            "strength": s, "method": method,
            "median_delta_log1p": float(np.median(d.delta_log1p)),
            "median_delta_jobs": float(np.median(d.delta_jobs)),
        })
sens = pd.DataFrame(sens_rows)
sens.to_csv(pipe_tbl("intervention_strength_sensitivity"), index=False)
sens

well-served median log1p = 9.619  (n_well_served = 518)


,strength,method,median_delta_log1p,median_delta_jobs
0,0.25,bayesian,0.772484,797.768830
1,0.25,deterministic,0.963615,516.744032
2,0.50,bayesian,1.544968,2524.928018
3,0.50,deterministic,1.927231,1871.191266
4,0.75,bayesian,2.317452,6264.363622
5,0.75,deterministic,2.890846,5421.384634


## 5 — Posterior predictive under intervention

Apply `delta_log1p` to every posterior draw for each target tract, then propagate a minor
queen-contiguous spillover (weight 0.1) to immediate neighbours. Compute baseline vs
intervention exceedance probabilities and a 95 % bootstrap CI on the exceedance change.

In [12]:
from src.modeling.spatial import adjacency_from_queen

tracts_sub = tracts_gdf[tracts_gdf.GEOID.isin(geoid_order)].copy()
if tracts_sub.crs is not None and getattr(tracts_sub.crs, "is_geographic", False):
    tracts_sub = tracts_sub.to_crs(3310)
W, w_geoids, _, _ = adjacency_from_queen(tracts_sub)
w_idx = {g: i for i, g in enumerate(w_geoids)}


def apply_intervention(delta_df: pd.DataFrame) -> np.ndarray:
    shifted = samples_wide.copy()
    target_set = set(delta_df.GEOID)
    target_idx = np.array([geoid_to_idx[g] for g in delta_df.GEOID])
    deltas = delta_df.delta_log1p.to_numpy()
    shifted[target_idx, :] += deltas[:, None]
    for g, d in zip(delta_df.GEOID, deltas):
        if g not in w_idx or d <= 0:
            continue
        nbr_cols = np.where(W[w_idx[g]] > 0)[0]
        for j in nbr_cols:
            g_nbr = w_geoids[j]
            if g_nbr in geoid_to_idx and g_nbr not in target_set:
                shifted[geoid_to_idx[g_nbr], :] += SPILLOVER_WEIGHT * d
    return shifted


shifted_A = apply_intervention(delta_A)
shifted_B = apply_intervention(delta_B)

baseline_exc = (samples_wide < Q25_LOG1P).mean(axis=1)
exc_A = (shifted_A < Q25_LOG1P).mean(axis=1)
exc_B = (shifted_B < Q25_LOG1P).mean(axis=1)

In [13]:
boot_A = np.empty((N_BOOT, n_tracts))
boot_B = np.empty((N_BOOT, n_tracts))
for b in range(N_BOOT):
    idx = rng.integers(0, n_draws, size=n_draws)
    ref = (samples_wide[:, idx] < Q25_LOG1P).mean(axis=1)
    boot_A[b] = (shifted_A[:, idx] < Q25_LOG1P).mean(axis=1) - ref
    boot_B[b] = (shifted_B[:, idx] < Q25_LOG1P).mean(axis=1) - ref

ci_A_low = np.quantile(boot_A, 0.025, axis=0)
ci_A_high = np.quantile(boot_A, 0.975, axis=0)
ci_B_low = np.quantile(boot_B, 0.025, axis=0)
ci_B_high = np.quantile(boot_B, 0.975, axis=0)
prob_improve_A = (boot_A < 0).mean(axis=0)
prob_improve_B = (boot_B < 0).mean(axis=0)

crossed_A = (baseline_exc > DESERT_EXCEEDANCE_THR) & (exc_A < DESERT_EXCEEDANCE_THR) & (prob_improve_A >= 0.80)
crossed_B = (baseline_exc > DESERT_EXCEEDANCE_THR) & (exc_B < DESERT_EXCEEDANCE_THR) & (prob_improve_B >= 0.80)

posterior_df = pd.DataFrame({
    "GEOID": geoid_order,
    "baseline_exceedance": baseline_exc,
    "exceedance_prob_scenario_A": exc_A,
    "exceedance_prob_scenario_B": exc_B,
    "exceedance_prob_delta_A": exc_A - baseline_exc,
    "exceedance_prob_delta_B": exc_B - baseline_exc,
    "exceedance_delta_A_ci_lower": ci_A_low,
    "exceedance_delta_A_ci_upper": ci_A_high,
    "exceedance_delta_B_ci_lower": ci_B_low,
    "exceedance_delta_B_ci_upper": ci_B_high,
    "prob_improve_A": prob_improve_A,
    "prob_improve_B": prob_improve_B,
    "targeted_by_A": np.isin(geoid_order, bayes_targets.GEOID.to_numpy()),
    "targeted_by_B": np.isin(geoid_order, det_targets.GEOID.to_numpy()),
    "crossed_threshold_A": crossed_A,
    "crossed_threshold_B": crossed_B,
})
posterior_df.to_csv(pipe_tbl("intervention_posterior"), index=False)
print(f"threshold crossings — A: {int(crossed_A.sum())}   B: {int(crossed_B.sum())}")

threshold crossings — A: 12   B: 8


## 6 — Equity impact

In [14]:
mm = master.merge(posterior_df, on="GEOID", how="inner").dropna(subset=["disadvantage_z"]).copy()

rho_base, _ = spearmanr(mm.baseline_exceedance, mm.disadvantage_z)
rho_A, _ = spearmanr(mm.exceedance_prob_scenario_A, mm.disadvantage_z)
rho_B, _ = spearmanr(mm.exceedance_prob_scenario_B, mm.disadvantage_z)
rho_base, rho_A, rho_B = float(rho_base), float(rho_A), float(rho_B)

pop = mm["pop_total"].fillna(0).astype(float)
pop_sum = float(pop.sum())
pop_cross_A = float(pop[mm.crossed_threshold_A].sum())
pop_cross_B = float(pop[mm.crossed_threshold_B].sum())

equity = pd.DataFrame([
    {"scenario": "baseline",         "spearman_exc_vs_disadv": rho_base, "n_crossings": 0,
     "pct_pop_moved_below_0p5": 0.0, "efficiency_crossings_per_target": 0.0},
    {"scenario": "A_bayesian",       "spearman_exc_vs_disadv": rho_A,    "n_crossings": int(mm.crossed_threshold_A.sum()),
     "pct_pop_moved_below_0p5": 100 * pop_cross_A / pop_sum if pop_sum else 0.0,
     "efficiency_crossings_per_target": float(mm.crossed_threshold_A.sum()) / N_INTERVENTION_TRACTS},
    {"scenario": "B_deterministic",  "spearman_exc_vs_disadv": rho_B,    "n_crossings": int(mm.crossed_threshold_B.sum()),
     "pct_pop_moved_below_0p5": 100 * pop_cross_B / pop_sum if pop_sum else 0.0,
     "efficiency_crossings_per_target": float(mm.crossed_threshold_B.sum()) / N_INTERVENTION_TRACTS},
])
equity["delta_spearman_vs_baseline"] = equity["spearman_exc_vs_disadv"] - rho_base
equity.to_csv(pipe_tbl("equity_impact"), index=False)
equity

,scenario,spearman_exc_vs_disadv,n_crossings,pct_pop_moved_below_0p5,efficiency_crossings_per_target,delta_spearman_vs_baseline
0,baseline,-0.466943,0,0.000000,0.0,0.000000
1,A_bayesian,-0.463499,12,1.987974,0.6,0.003444
2,B_deterministic,-0.460010,8,1.084713,0.4,0.006933


In [15]:
if HOOK_GEOID in geoid_to_idx:
    i = geoid_to_idx[HOOK_GEOID]
    hook = pd.DataFrame([{
        "GEOID": HOOK_GEOID,
        "baseline_exceedance": float(baseline_exc[i]),
        "scenario_A_exceedance": float(exc_A[i]),
        "scenario_A_delta": float(exc_A[i] - baseline_exc[i]),
        "scenario_A_ci_lower": float(ci_A_low[i]),
        "scenario_A_ci_upper": float(ci_A_high[i]),
        "scenario_A_prob_improve": float(prob_improve_A[i]),
        "targeted_by_bayes": HOOK_GEOID in bayes_targets.GEOID.tolist(),
        "targeted_by_det": HOOK_GEOID in det_targets.GEOID.tolist(),
    }])
    hook.to_csv(pipe_tbl("hook_tract_intervention"), index=False)
    hook

## 7 — Deterministic vs Bayesian classification

In [16]:
det_desert = master.det_jobs < Q25_JOBS
bayes_desert = master.exceedance_prob > DESERT_EXCEEDANCE_THR
cls = pd.DataFrame({
    "GEOID": master.GEOID,
    "det_jobs": master.det_jobs,
    "exceedance_prob": master.exceedance_prob,
    "disadvantage_z": master.disadvantage_z,
    "det_desert": det_desert.to_numpy(),
    "bayes_desert": bayes_desert.to_numpy(),
})
cls["classification"] = "TN"
cls.loc[det_desert & bayes_desert, "classification"] = "TP"
cls.loc[det_desert & (~bayes_desert), "classification"] = "FP"
cls.loc[(~det_desert) & bayes_desert, "classification"] = "FN"
cls.to_csv(pipe_tbl("det_vs_bayes_classification"), index=False)
counts = cls["classification"].value_counts().to_dict()
counts

{'TN': 540, 'TP': 178, 'FP': 2}

## 8 — Fig 1 (study area) + Fig 9 (intervention)

In [17]:
nctd_stops_path = REPO_ROOT / "data" / "raw" / "gtfs" / "nctd" / "extracted" / "stops.txt"
nctd_stops = pd.read_csv(nctd_stops_path, dtype=str, low_memory=False)
nctd_stops["stop_lon"] = pd.to_numeric(nctd_stops["stop_lon"])
nctd_stops["stop_lat"] = pd.to_numeric(nctd_stops["stop_lat"])
nctd_gdf = gpd.GeoDataFrame(
    nctd_stops, geometry=gpd.points_from_xy(nctd_stops.stop_lon, nctd_stops.stop_lat), crs="EPSG:4326"
)

tracts_plot = tracts_gdf.to_crs(3857)
mts_plot = stops_gdf.to_crs(3857)
nctd_plot = nctd_gdf.to_crs(3857)

fig, ax = plt.subplots(figsize=(9, 10))
tracts_plot.plot(ax=ax, facecolor="#f7f7f7", edgecolor="#bbb", linewidth=0.3)
mts_plot.plot(ax=ax, color="#1f77b4", markersize=1, label=f"MTS stops (n={len(mts_plot)})")
nctd_plot.plot(ax=ax, color="#ff7f0e", markersize=1, label=f"NCTD stops (n={len(nctd_plot)})")
if HOOK_GEOID in tracts_plot.GEOID.values:
    tracts_plot[tracts_plot.GEOID == HOOK_GEOID].plot(
        ax=ax, facecolor="#d62728", alpha=0.55, edgecolor="#7a0e15", linewidth=0.6,
    )
ax.legend(loc="upper right", frameon=True)
ax.set_title(f"San Diego County study area — {len(tracts_plot)} tracts, MTS + NCTD stops")
ax.set_axis_off()
fig.tight_layout()
fig1_path = pipe_fig("study_area_map", with_rid=False)
fig.savefig(fig1_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"wrote {fig1_path.relative_to(REPO_ROOT)}")

wrote artifacts\figures\pipeline__07_study_area_map.png


In [18]:
merged = tracts_gdf.merge(posterior_df, on="GEOID", how="inner").to_crs(3857)

fig, axes = plt.subplots(1, 3, figsize=(18, 7), gridspec_kw={"width_ratios": [1, 1, 0.55]})

ax = axes[0]
merged.plot(column="baseline_exceedance", cmap="viridis", ax=ax, legend=True,
            legend_kwds={"label": "P(jobs < Q25 | data)", "shrink": 0.6},
            linewidth=0.1, edgecolor="#888")
ax.set_title("A — Baseline exceedance probability")
ax.set_axis_off()

ax = axes[1]
delta_vals = merged["exceedance_prob_delta_A"].to_numpy()
vmin = min(-0.01, float(delta_vals.min()))
vmax = max(0.01, float(delta_vals.max()))
norm = TwoSlopeNorm(vcenter=0.0, vmin=vmin, vmax=vmax)
merged.plot(column="exceedance_prob_delta_A", cmap="RdYlGn_r", norm=norm, ax=ax, legend=True,
            legend_kwds={"label": "Δ exceedance (Scenario A)", "shrink": 0.6},
            linewidth=0.1, edgecolor="#888")
starred = merged[merged.crossed_threshold_A]
if len(starred):
    starred.assign(geometry=starred.geometry.centroid).plot(
        ax=ax, color="black", marker="*", markersize=120, zorder=5,
        label=f"Crossed P<0.5 (n={len(starred)})"
    )
det_only_plot = merged[merged.GEOID.isin(only_det)]
if len(det_only_plot):
    det_only_plot.assign(geometry=det_only_plot.geometry.centroid).plot(
        ax=ax, facecolors="none", edgecolors="#222", marker="o", markersize=80,
        linewidth=1.4, zorder=6,
        label=f"Det-only targets (n={len(det_only_plot)})"
    )
ax.legend(loc="lower left", fontsize=9, frameon=True)
ax.set_title("B — Δ exceedance, Bayesian targeting")
ax.set_axis_off()

ax = axes[2]
n_A, n_B = int(crossed_A.sum()), int(crossed_B.sum())
bars = ax.bar(["A: Bayesian", "B: Deterministic"], [n_A, n_B],
              color=["#2a9d8f", "#e76f51"])
for rect, val in zip(bars, [n_A, n_B]):
    ax.text(rect.get_x() + rect.get_width() / 2, val + 0.1, str(val),
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("Threshold crossings (P>0.5 → P<0.5, 80% certain)")
ax.set_title("C — Efficiency comparison")
ax.set_ylim(0, max([n_A, n_B, 1]) * 1.25 + 1)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"Intervention simulation — {N_INTERVENTION_TRACTS} tracts, strength={INTERVENTION_STRENGTH} "
    f"(ρ baseline={rho_base:.3f}, ρ_A={rho_A:.3f}, ρ_B={rho_B:.3f})",
    fontsize=12,
)
fig.tight_layout()
fig9_path = pipe_fig("intervention_fig9")
fig.savefig(fig9_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"wrote {fig9_path.relative_to(REPO_ROOT)}")

wrote artifacts\figures\pipeline__07_intervention_fig9__fit_raw_zscore_x.png


## 9 — Dashboard GeoJSON export (INTERFACES.md contract)

In [19]:
wass_path = PIPE_TABLES / f"pipeline__05_wasserstein_tract__{RID}.csv"
wasserstein = pd.read_csv(wass_path, dtype={"GEOID": str})
wasserstein["GEOID"] = wasserstein["GEOID"].str.zfill(11)
wass_col = [c for c in wasserstein.columns if c != "GEOID"][0]
wasserstein = wasserstein.rename(columns={wass_col: "wasserstein_dist"})

base_props = (
    post_summary[["GEOID", "posterior_mean_log1p", "posterior_sd_log1p",
                   "ci_lower_95_log1p", "ci_upper_95_log1p",
                   "posterior_mean_jobs", "exceedance_prob_45min", "p_transit_desert"]]
    .merge(wasserstein, on="GEOID", how="left")
    .merge(acs[["GEOID", "NAME", "B01003_001E", "no_vehicle_hh_rate",
                "B19013_001E", "poverty_rate"]], on="GEOID", how="left")
    .rename(columns={
        "NAME": "tract_name",
        "B01003_001E": "pop_total",
        "no_vehicle_hh_rate": "pct_no_vehicle",
        "B19013_001E": "median_income",
        "poverty_rate": "pct_poverty",
        "exceedance_prob_45min": "exceedance_prob",
        "posterior_mean_log1p": "posterior_mean",
        "posterior_sd_log1p": "posterior_sd",
        "ci_lower_95_log1p": "ci_lower_95",
        "ci_upper_95_log1p": "ci_upper_95",
    })
)
base_props["entropy"] = np.nan

baseline_gdf = (
    tracts_gdf[["GEOID", "geometry"]]
    .merge(base_props, on="GEOID", how="inner")
    .rename(columns={"GEOID": "geoid"})
)
baseline_path = GEOJSON_DIR / "sd_tracts_equity_baseline.geojson"
if baseline_path.exists():
    baseline_path.unlink()
baseline_gdf.to_file(baseline_path, driver="GeoJSON")
print(f"baseline: {len(baseline_gdf)} features → {baseline_path.relative_to(REPO_ROOT)}")

baseline: 720 features → data\processed\geojson\sd_tracts_equity_baseline.geojson


In [20]:
def write_scenario_geojson(scenario_id: str, scen_df: pd.DataFrame) -> Path:
    gdf = baseline_gdf.merge(
        scen_df[["GEOID", "exceedance_prob_scenario", "exceedance_prob_delta",
                  "p_transit_desert_scenario", "crossed_threshold",
                  "exceedance_delta_ci_lower", "exceedance_delta_ci_upper"]]
        .rename(columns={"GEOID": "geoid"}),
        on="geoid", how="left",
    )
    path = SCENARIO_DIR / f"{scenario_id}.geojson"
    if path.exists():
        path.unlink()
    gdf.to_file(path, driver="GeoJSON")
    return path


scen_A_df = posterior_df.rename(columns={
    "exceedance_prob_scenario_A": "exceedance_prob_scenario",
    "exceedance_prob_delta_A": "exceedance_prob_delta",
    "exceedance_delta_A_ci_lower": "exceedance_delta_ci_lower",
    "exceedance_delta_A_ci_upper": "exceedance_delta_ci_upper",
    "crossed_threshold_A": "crossed_threshold",
}).copy()
scen_A_df["p_transit_desert_scenario"] = scen_A_df["exceedance_prob_scenario"]

scen_B_df = posterior_df.rename(columns={
    "exceedance_prob_scenario_B": "exceedance_prob_scenario",
    "exceedance_prob_delta_B": "exceedance_prob_delta",
    "exceedance_delta_B_ci_lower": "exceedance_delta_ci_lower",
    "exceedance_delta_B_ci_upper": "exceedance_delta_ci_upper",
    "crossed_threshold_B": "crossed_threshold",
}).copy()
scen_B_df["p_transit_desert_scenario"] = scen_B_df["exceedance_prob_scenario"]

scen_A_path = write_scenario_geojson(SCENARIO_A_ID, scen_A_df)
scen_B_path = write_scenario_geojson(SCENARIO_B_ID, scen_B_df)

manifest = {
    "run_id": RID,
    "generated": TODAY,
    "baseline_geojson": baseline_path.name,
    "scenarios": [
        {
            "id": SCENARIO_A_ID,
            "label": "Frequency Doubling — Bayesian Targets",
            "description": f"Simulated {int(INTERVENTION_STRENGTH * 100)}% accessibility-gap closure for the {N_INTERVENTION_TRACTS} highest-exceedance tracts",
            "n_tracts_intervened": N_INTERVENTION_TRACTS,
            "n_threshold_crossings": int(crossed_A.sum()),
            "efficiency_ratio_vs_B": float(crossed_A.sum()) / max(int(crossed_B.sum()), 1),
        },
        {
            "id": SCENARIO_B_ID,
            "label": "Frequency Doubling — Deterministic Targets",
            "description": f"Same intervention applied to the {N_INTERVENTION_TRACTS} lowest deterministic-jobs tracts",
            "n_tracts_intervened": N_INTERVENTION_TRACTS,
            "n_threshold_crossings": int(crossed_B.sum()),
        },
    ],
    "key_tracts": [
        {"geoid": HOOK_GEOID, "label": "Hook tract (coin flip)",
         "exceedance_baseline": float(baseline_exc[geoid_to_idx[HOOK_GEOID]]) if HOOK_GEOID in geoid_to_idx else None,
         "exceedance_scenario_A": float(exc_A[geoid_to_idx[HOOK_GEOID]]) if HOOK_GEOID in geoid_to_idx else None},
    ],
}
with open(GEOJSON_DIR / "dashboard_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("manifest written")

manifest written


In [21]:
hook_idx = geoid_to_idx.get(HOOK_GEOID)
hook_base = float(baseline_exc[hook_idx]) if hook_idx is not None else float("nan")
hook_A = float(exc_A[hook_idx]) if hook_idx is not None else float("nan")
hook_A_lo = float(ci_A_low[hook_idx]) if hook_idx is not None else float("nan")
hook_A_hi = float(ci_A_high[hook_idx]) if hook_idx is not None else float("nan")

rows = [
    ("targeting", "n_overlap", "both", len(overlap), None, None,
     "Tracts in both Bayesian and deterministic target lists"),
    ("targeting", "n_bayes_only", "A", len(only_bayes), None, None,
     "Bayesian-only targets (likely false negatives of deterministic)"),
    ("targeting", "n_det_only", "B", len(only_det), None, None,
     "Deterministic-only targets (likely false positives)"),
    ("intervention", "n_threshold_crossings", "A", int(crossed_A.sum()), None, None,
     "Tracts P:>0.5→<0.5 at ≥80% posterior certainty (Scenario A)"),
    ("intervention", "n_threshold_crossings", "B", int(crossed_B.sum()), None, None,
     "Tracts P:>0.5→<0.5 at ≥80% posterior certainty (Scenario B)"),
    ("intervention", "pct_pop_moved", "A",
     100 * pop_cross_A / pop_sum if pop_sum else 0.0, None, None,
     "Share of population moved below 0.5 exceedance (A)"),
    ("intervention", "pct_pop_moved", "B",
     100 * pop_cross_B / pop_sum if pop_sum else 0.0, None, None,
     "Share of population moved below 0.5 exceedance (B)"),
    ("equity", "spearman", "baseline", rho_base, None, None,
     "Spearman(exceedance, disadvantage_z) pre-intervention"),
    ("equity", "spearman", "A", rho_A, None, None, "Scenario A post-intervention"),
    ("equity", "spearman", "B", rho_B, None, None, "Scenario B post-intervention"),
    ("equity", "delta_spearman", "A", rho_A - rho_base, None, None, None),
    ("equity", "delta_spearman", "B", rho_B - rho_base, None, None, None),
    ("classification", "count_TP", "global", counts.get("TP", 0), None, None,
     "Det low + Bayes high exceedance"),
    ("classification", "count_FP", "global", counts.get("FP", 0), None, None,
     "Det flags as desert, Bayes disagrees (wasted targeting)"),
    ("classification", "count_FN", "global", counts.get("FN", 0), None, None,
     "Bayes flags as desert, Det misses (hook-like tracts)"),
    ("classification", "count_TN", "global", counts.get("TN", 0), None, None, None),
    ("hook", "exceedance_baseline", HOOK_GEOID, hook_base, None, None, None),
    ("hook", "exceedance_scenario_A", HOOK_GEOID, hook_A, hook_A_lo, hook_A_hi, None),
    ("calibration", "spearman_headway_jobs", "all_tracts", rho_head, None, None,
     "AM-peak min headway vs det_jobs"),
]
summary = pd.DataFrame(rows, columns=["analysis_type", "metric", "scenario",
                                      "value", "ci_lower", "ci_upper", "note"])
summary.to_csv(pipe_tbl("summary"), index=False)
summary

,analysis_type,metric,scenario,value,ci_lower,ci_upper,note
0,targeting,n_overlap,both,6.000000,NaN,NaN,Tracts in both Bayesian and deterministic targ...
1,targeting,n_bayes_only,A,14.000000,NaN,NaN,Bayesian-only targets (likely false negatives ...
2,targeting,n_det_only,B,14.000000,NaN,NaN,Deterministic-only targets (likely false posit...
3,intervention,n_threshold_crossings,A,12.000000,NaN,NaN,Tracts P:>0.5→<0.5 at ≥80% posterior certainty...
4,intervention,n_threshold_crossings,B,8.000000,NaN,NaN,Tracts P:>0.5→<0.5 at ≥80% posterior certainty...
5,intervention,pct_pop_moved,A,1.987974,NaN,NaN,Share of population moved below 0.5 exceedance...
6,intervention,pct_pop_moved,B,1.084713,NaN,NaN,Share of population moved below 0.5 exceedance...
7,equity,spearman,baseline,-0.466943,NaN,NaN,"Spearman(exceedance, disadvantage_z) pre-inter..."
8,equity,spearman,A,-0.463499,NaN,NaN,Scenario A post-intervention
9,equity,spearman,B,-0.460010,NaN,NaN,Scenario B post-intervention
